# Specification Analysis Compliance Calculation

This notebook analyzes selected parametric specifications, calculates compliance from associated measurements, updates spec properties, and uploads an Excel report to the product.


### Imports

Import the necessary Python modules to execute the notebook. [**`nisystemlink.clients`**](https://github.com/ni/nisystemlink-clients-python) provides the predefined models and methods for `Specification`, `Test Monitor`, `Product` and `File` APIs. **`Scrapbook`** is used to run notebooks and record data for integration with the SystemLink Notebook Execution Service. **`Pandas`** and **`NumPy`** are used for building dataframes and calculating compliance. **`ni_unit_converter`** is used for unit conversion.

In [ ]:
import io
import json
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import scrapbook as sb
from ni_unit_converter import convert_from_base_unit, convert_to_base_unit
from nisystemlink.clients.file import FileClient
from nisystemlink.clients.product import ProductClient
from nisystemlink.clients.product.models import UpdateProductRequest
from nisystemlink.clients.spec import SpecClient
from nisystemlink.clients.spec.models import (
    NumericConditionValue,
    QuerySpecificationsRequest,
    Specification,
    SpecificationLimit,
    SpecificationType,
    StringConditionValue,
    UpdateSpecificationsRequest,
)
from nisystemlink.clients.testmonitor import TestMonitorClient
from nisystemlink.clients.testmonitor.models import QueryStepsRequest, Step, StepProjection


### Input Parameters

1. `spec_ids` - IDs of the specifications.
1. `product_id` - ID of the product that contains the specifications and test results.

These are the parameters that the notebook expects to be passed in by SystemLink. For notebooks designed to be triggered by pressing the 'Analyze' button in SystemLink Specs grid inside product details page, they must tag the cell with 'parameters' and at minimum specify the following in the cell metadata using the JupyterLab Property Inspector (double gear icon):

```json
{
  "papermill": {
    "parameters": {
      "spec_ids": [],
      "product_id": ""
    }
  },
  "systemlink": {
    "namespaces": [],
    "parameters": [
      {
        "display_name": "spec_ids",
        "id": "spec_ids",
        "type": "string[]"
      },
      {
        "display_name": "product_id",
        "id": "product_id",
        "type": "string"
      }
    ],
    "version": 2
  },
  "tags": ["parameters"]
}
```

For more information on how parameterization works, review the [papermill documentation](https://papermill.readthedocs.io/en/latest/usage-parameterize.html#how-parameters-work).

In [ ]:
spec_ids = [""]
product_id = ""

### Constants

In [ ]:
QUERY_SPECS_TAKE_COUNT = 1000
QUERY_STEPS_TAKE_COUNT = 1000
EXCEL_FILE_NAME = "Spec_Compliance.xlsx"

### Data fields and compliance properties

In [ ]:
EMPTY_CATEGORY = "Empty"


class DataframeHeaders:
    SPEC_DETAILS = "Spec Details"
    SPEC_ID = "SpecID"
    CATEGORY = "Category"
    BLOCK = "Block"
    SYMBOL = "Symbol"
    NAME = "Name"
    MIN = "Min"
    TYPICAL = "Typical"
    MAX = "Max"
    UNIT = "Unit"
    DISCRETE = "Discrete"
    STEP = "Step"
    CONDITION = "Condition"
    COMPLIANCE = "Compliance"
    CP = "Cp"
    CPK = "Cpk"
    STANDARD_DEVIATION = "Standard Deviation"


class ComplianceParameters:
    MIN = "Min"
    MAX = "Max"
    MEAN = "Mean"
    MEDIAN = "Median"
    HEALTH = "Health"
    CP = "Cp"
    CPK = "Cpk"
    STANDARD_DEVIATION = "Standard Deviation"


class SpecComplianceProperties:
    MIN = "Compliance Min"
    MAX = "Compliance Max"
    MEAN = "Compliance Mean"
    MEDIAN = "Compliance Median"
    HEALTH = "Compliance Health"
    CP = "Compliance Cp"
    CPK = "Compliance Cpk"
    STANDARD_DEVIATION = "Compliance Standard Deviation"


class SpecHealth:
    PASS = "Pass"
    FAIL = "Fail"
    INF = "inf"
    NEGATIVE_INF = "-inf"
    NAN = "nan"

MAX_SPECS_PER_UPDATE_REQUEST = 100

### Error messages

In [ ]:
class ErrorMessages:
    MISMATCHING_BASE_UNITS = "Input and output units are having different base unit"

### Python clients

Initialize `SpecClient`, `TestMonitorClient`, `ProductClient` and `FileClient` to access the respective APIs.

In [ ]:
spec_client = SpecClient()
test_monitor_client = TestMonitorClient()
product_client = ProductClient()
file_client = FileClient()

### API functions

Functions that call the SystemLink APIs through the Python clients.

In [ ]:
# Fetches the selected parametric specs of the product, including their measurement mapping.
def query_parametric_specs(product_id: str) -> List[Specification]:
    request = QuerySpecificationsRequest(
        product_ids=[product_id],
        filter=f'(({__generate_spec_ids_filter()}) && (type == "{SpecificationType.PARAMETRIC.value}"))',
        take=QUERY_SPECS_TAKE_COUNT,
    )
    specs = []
    response = spec_client.query_specs(request)
    while True:
        specs.extend(response.specs or [])
        if not response.continuation_token:
            break
        request.continuation_token = response.continuation_token
        response = spec_client.query_specs(request)
    return specs


# Builds the OR filter that selects the input spec IDs.
def __generate_spec_ids_filter() -> str:
    return " || ".join(f'specId == "{spec_id}"' for spec_id in spec_ids)


# Fetches the test steps and their measurements that match the given step and result filters.
def query_steps(result_filter: str | None, filter: str | None) -> List[Step]:
    request = QueryStepsRequest(
        filter=filter,
        result_filter=result_filter,
        take=QUERY_STEPS_TAKE_COUNT,
        projection=[StepProjection.DATA, StepProjection.INPUTS],
    )
    steps = []
    response = test_monitor_client.query_steps(request)
    while True:
        steps.extend(response.steps or [])
        if not response.continuation_token:
            break
        request.continuation_token = response.continuation_token
        response = test_monitor_client.query_steps(request)
    return steps


# Saves the specs, with the compliance properties.
def update_specs(updated_specs: List[Specification]):
    request = UpdateSpecificationsRequest(
        specs=[spec.model_dump(mode="json", by_alias=True, exclude_unset=True) for spec in updated_specs]
    )
    return spec_client.update_specs(request)


# Uploads the Excel report to the File service and returns the file ID.
def upload_file(file_name: str, workspace_id: str) -> str:
    file_stream = io.BytesIO(Path(file_name).read_bytes())
    file_stream.name = f"{part_number}-{Path(file_name).name}"
    try:
        return file_client.upload_file(file_stream, workspace=workspace_id)
    finally:
        file_stream.close()


# Attaches the uploaded report file to the product.
def add_file_id_to_product(product_information, file_id: str):
    file_ids = list(product_information.file_ids or [])
    file_ids.append(file_id)
    request = UpdateProductRequest(id=product_information.id, file_ids=file_ids)
    return product_client.update_products([request], replace=False)


### Get product information

Retrieve the product's part number and workspace. The part number scopes measurement results, and the workspace is used when uploading the report.


In [ ]:
product_information = product_client.get_product(product_id)
part_number = product_information.part_number

### Select measurement filters

Choose the query filters for each spec's measurements.

1. If the spec has a measurement mapping, its `filter` and `resultFilter` are used. An empty `filter` means no step filter. An empty `resultFilter` falls back to the default.
1. If the spec has no measurement mapping, the defaults are used:
   - `filter`: the measurement name is the spec name, or its SpecID is the spec ID.
   - `resultFilter`: the results belong to the product's part number.
1. The returned steps are filtered again, so only measurements whose name or SpecID equals a value in the filter (matched with OR) are kept. If the filter has no such conditions, all returned measurements are kept.

In [ ]:
# Returns the result filter and step filter of a spec, from its measurement mapping or the defaults.
def get_steps_filters(spec: Specification, part_number: str) -> Tuple[str | None, str | None]:
    default_result_filter = f'(partNumber == "{part_number}")'
    if not spec.measurement_mapping:
        default_filter = (
            f'((data.parameters.Any(it["name"] = "{spec.name}"))'
            f' || (data.parameters.Any(it["SpecID"] = "{spec.spec_id}")))'
        )
        return default_result_filter, default_filter

    mapping = spec.measurement_mapping[0]
    return mapping.result_filter or default_result_filter, mapping.filter


# Reads the measurement names and SpecIDs from the equals conditions of a step filter.
def get_mapping_parameter_filter(steps_filter: str | None) -> Tuple[set, set]:
    names = set()
    spec_ids = set()
    pattern = r'data\.parameters\.Any\(\s*it\["(name|SpecID)"\]\s*==?\s*"((?:[^"\\]|\\.)*)"\s*\)'
    for field, value in re.findall(pattern, steps_filter or ""):
        value = re.sub(r'\\(.)', r'\1', value)
        (names if field == "name" else spec_ids).add(value)
    return names, spec_ids


### Value and unit helpers

Provides methods to convert values and units.

In [ ]:
# Converts a string to a Decimal, or None if it is not a number.
def __convert_to_decimal(value: str) -> Decimal | None:
    try:
        return Decimal(value)
    except Exception:
        return None


# Converts a 'true' or 'false' string to a bool, or None for any other value.
def __convert_to_bool(value: str) -> bool | None:
    if value in ["True", "true"]:
        return True
    if value in ["False", "false"]:
        return False
    return None


# Converts a string to a bool or a Decimal when possible, else keeps the string.
def __convert_to_datatype(value: str) -> bool | Decimal | str:
    bool_value = __convert_to_bool(value=value)
    if bool_value is not None:
        return bool_value

    decimal_value = __convert_to_decimal(value=value)
    if decimal_value is not None:
        return decimal_value

    return value


# Splits a condition string such as 'Vcc (V)' into its name and unit.
def extract_condition_name_and_unit(condition_string: str) -> Tuple[str, str | None]:
    pattern = r'^(.*?)\((.*?)\)$'
    match = re.match(pattern, condition_string)
    if match:
        condition_name = match.group(1).strip()
        condition_unit = match.group(2).strip()

        return condition_name, condition_unit

    return condition_string.strip(), None


# Checks whether two units have different base units.
def is_different_base_unit(input_unit: str, output_unit: str) -> bool:
    input_base_unit = convert_to_base_unit(input_unit, 1).base_unit
    output_base_unit = convert_to_base_unit(output_unit, 1).base_unit

    if input_base_unit != output_base_unit:
        return True

    return False


# Converts a value from one unit to another unit with the same base unit.
def convert_unit(input_unit: str, output_unit: str, value: str) -> Decimal:
    if is_different_base_unit(input_unit, output_unit):
        raise Exception(ErrorMessages.MISMATCHING_BASE_UNITS)
    output_value = convert_to_base_unit(input_unit, value).converted_value
    output_value = convert_from_base_unit(output_unit, output_value)

    return output_value


### Condition mapping and measurement selection

Provides methods to map the step conditions to the spec conditions and to select the measurements of a spec.

#### Rules for condition mapping

1. If the condition value in the measurement data is empty, it is considered within bounds.
1. If the condition value is within the range or the discrete values of the spec condition, it is considered within bounds.

#### Rules for measurement data

1. If the measurement value is empty, it is not considered for compliance.
1. If the measurement value is not a valid number, it is not considered for compliance.

In [ ]:
# Returns the spec conditions as a dictionary of condition name and value.
def get_spec_condition(spec: Specification) -> Dict[str, NumericConditionValue | StringConditionValue]:
    return {condition.name: condition.value for condition in (spec.conditions or [])}


# Returns the step inputs that match the spec conditions, converted to the spec condition units.
def get_step_condition_matching_the_spec_condition(
    step: Step,
    spec_condition: Dict[str, NumericConditionValue | StringConditionValue]
) -> Dict:
    step_condition = {}

    for step_input in step.inputs or []:
        step_condition_name, step_condition_unit = extract_condition_name_and_unit(step_input.name)

        if step_condition_name not in spec_condition:
            continue

        spec_condition_unit = getattr(spec_condition[step_condition_name], "unit", None)
        if spec_condition_unit is None and step_condition_unit:
            continue

        has_value = "value" in step_input.model_fields_set

        if (step_condition_unit == spec_condition_unit) or (
            spec_condition_unit and not step_condition_unit
        ):
            try:
                step_condition[step_condition_name] = (
                    Decimal(str(step_input.value)) if has_value else ''
                )
            except InvalidOperation:
                step_condition[step_condition_name] = step_input.value
        else:
            if is_different_base_unit(step_condition_unit, spec_condition_unit):
                continue
            if not has_value or step_input.value == str():
                step_condition[step_condition_name] = str()
            else:
                try:
                    step_condition_value = convert_unit(
                        step_condition_unit, spec_condition_unit, str(step_input.value)
                    )
                    step_condition[step_condition_name] = step_condition_value
                except (ValueError, InvalidOperation):
                    step_condition[step_condition_name] = step_input.value

    return step_condition


# Checks whether a step condition value is within a spec condition range.
def check_step_condition_within_spec_condition_range(
    spec_condition: Dict[str, NumericConditionValue | StringConditionValue],
    condition_name: str,
    condition_value: Decimal
) -> bool:
    for spec_condition_range in getattr(spec_condition[condition_name], "range", None) or []:
        min_limit = __convert_to_decimal(str(spec_condition_range.min))
        max_limit = __convert_to_decimal(str(spec_condition_range.max))
        if (min_limit is None or condition_value >= min_limit) and (
            max_limit is None or condition_value <= max_limit
        ):
            return True

    return False


# Checks whether a step condition value is one of the spec condition discrete values.
def check_step_condition_within_spec_condition_discrete_array(
    spec_condition: Dict[str, NumericConditionValue | StringConditionValue],
    condition_name: str,
    condition_value: Decimal | str
) -> bool:
    discrete_values = getattr(spec_condition[condition_name], "discrete", None) or []
    converted_discrete_values = [
        __convert_to_datatype(str(discrete_value)) for discrete_value in discrete_values
    ]

    return condition_value in converted_discrete_values


# Checks whether all the step condition values are within the spec conditions.
def check_step_condition_within_spec_condition(spec_condition: Dict, step_condition: Dict) -> bool:
    condition_within_range = True

    for condition_name, condition_value in step_condition.items():
        try:
            condition_value = Decimal(str(condition_value))
        except InvalidOperation:
            pass

        if not condition_value and isinstance(condition_value, str):
            continue

        condition_within_range = (
            (
                check_step_condition_within_spec_condition_range(
                    spec_condition, condition_name, condition_value
                )
                or check_step_condition_within_spec_condition_discrete_array(
                    spec_condition, condition_name, condition_value
                )
            )
            if isinstance(condition_value, Decimal)
            else check_step_condition_within_spec_condition_discrete_array(
                spec_condition, condition_name, condition_value
            )
        )

        if not condition_within_range:
            return condition_within_range

    return condition_within_range


# Returns the valid measurement values of a spec, converted to the spec unit, after condition mapping.
def fetch_measurement_data_for_spec(
    steps: List[Step],
    steps_filter: str | None,
    spec_condition: Dict[str, NumericConditionValue | StringConditionValue],
    spec_unit: str | None
) -> List:
    measurement_data = []
    filter_names, filter_spec_ids = get_mapping_parameter_filter(steps_filter)

    for step in steps:
        step_condition = get_step_condition_matching_the_spec_condition(step, spec_condition)
        if not check_step_condition_within_spec_condition(spec_condition, step_condition):
            continue
        if not step.data or step.data.parameters is None:
            continue
        for parameter in step.data.parameters:
            step_unit = parameter.units or None
            if (spec_unit and not step_unit) or (step_unit and not spec_unit):
                continue
            if spec_unit and step_unit and is_different_base_unit(step_unit, spec_unit):
                continue
            if (filter_names or filter_spec_ids) and not (
                parameter.name in filter_names
                or (parameter.model_extra or {}).get("SpecID") in filter_spec_ids
            ):
                continue
            measurement_value = parameter.measurement or str()
            if measurement_value:
                try:
                    if not spec_unit:
                        measurement_data.append(Decimal(str(measurement_value)))
                    else:
                        measurement_value = convert_from_base_unit(spec_unit, measurement_value)
                        measurement_data.append(measurement_value)
                except (TypeError, ValueError, InvalidOperation):
                    continue

    return measurement_data


### Compliance metrics

Provides methods to calculate the min, max, mean, median, Cp, Cpk and health of a spec.

In [ ]:
# Calculates the min, max, mean or median of the measurement data.
def calculate_compliance(measurement_data: pd.DataFrame, parameter: str) -> pd.Series:
    if parameter == ComplianceParameters.MIN:
        return measurement_data.min(skipna=True)
    if parameter == ComplianceParameters.MAX:
        return measurement_data.max(skipna=True)
    if parameter == ComplianceParameters.MEAN:
        return measurement_data.mean(skipna=True)
    if parameter == ComplianceParameters.MEDIAN:
        return measurement_data.median(skipna=True)
    return pd.Series()


# Calculates the upper Cpk from the high limit, mean and standard deviation.
def __calculate_cpk_high(high_limit, mean, std):
    diff = (high_limit) - (mean)
    if std == 0:
        return np.inf
    return diff / (3 * (std))


# Calculates the lower Cpk from the low limit, mean and standard deviation.
def __calculate_cpk_low(low_limit, mean, std):
    diff = (mean) - (low_limit)
    if std == 0:
        return np.inf
    return diff / (3 * (std))


# Calculates Cp from the high limit, low limit and standard deviation.
def __calculate_cp(high_limit, low_limit, std):
    diff = (high_limit) - (low_limit)
    if std == 0:
        return np.inf
    return diff / (6 * (std))


# Returns the spec health (Pass or Fail) from the min and max compliance and the spec limits.
def get_spec_health(spec: Specification, min_compliance: str, max_compliance: str) -> str | None:
    limit = spec.limit or SpecificationLimit()
    if spec.unit is not None:
        spec_min = (
            convert_to_base_unit(spec.unit, limit.min).converted_value
            if limit.min is not None
            else Decimal(SpecHealth.NEGATIVE_INF)
        )
        spec_max = (
            convert_to_base_unit(spec.unit, limit.max).converted_value
            if limit.max is not None
            else Decimal(SpecHealth.INF)
        )
        min_compliance = convert_to_base_unit(spec.unit, min_compliance).converted_value
        max_compliance = convert_to_base_unit(spec.unit, max_compliance).converted_value
    else:
        spec_min = (
            Decimal(str(limit.min))
            if limit.min is not None
            else Decimal(SpecHealth.NEGATIVE_INF)
        )
        spec_max = (
            Decimal(str(limit.max))
            if limit.max is not None
            else Decimal(SpecHealth.INF)
        )
        min_compliance = Decimal(str(min_compliance))
        max_compliance = Decimal(str(max_compliance))

    try:
        if spec_min is Decimal(SpecHealth.NEGATIVE_INF) and spec_max is not None:
            return SpecHealth.PASS if max_compliance <= spec_max else SpecHealth.FAIL
        if spec_max is Decimal(SpecHealth.INF) and spec_min is not None:
            return SpecHealth.PASS if min_compliance <= spec_min else SpecHealth.FAIL
        if spec_min is Decimal(SpecHealth.NEGATIVE_INF) and spec_max is Decimal(SpecHealth.INF):
            return SpecHealth.PASS
        if spec_min <= min_compliance <= spec_max and spec_min <= max_compliance <= spec_max:
            return SpecHealth.PASS
        return SpecHealth.FAIL
    except InvalidOperation:
        return None


### Dataframes

Provides methods to build the spec details, conditions and compliance dataframes of the report.

In [ ]:
# Concatenates dataframes, optionally resetting their index first.
def concatenate_dataframes(
    dfs: List[pd.DataFrame],
    reset_index: bool = True,
    axis: int = 1
) -> pd.DataFrame:
    if reset_index:
        for df in dfs:
            df.reset_index(drop=True, inplace=True)
    concatenated_df = pd.concat(dfs, axis=axis)

    return concatenated_df


# Builds the spec details dataframe of a spec.
def generate_spec_details_dataframe(spec: Specification) -> pd.DataFrame:
    limit = spec.limit or SpecificationLimit()
    spec_df = pd.DataFrame(
        {
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.SPEC_ID): [spec.spec_id],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.CATEGORY): [spec.category],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.BLOCK): [spec.block],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.SYMBOL): [spec.symbol],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.NAME): [spec.name],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.MIN): [limit.min],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.TYPICAL): [limit.typical],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.MAX): [limit.max],
            (DataframeHeaders.SPEC_DETAILS, DataframeHeaders.UNIT): [spec.unit],
        }
    )

    return spec_df


# Builds the spec conditions dataframe of a spec.
def generate_spec_conditions_dataframe(
    spec_condition: Dict[str, NumericConditionValue | StringConditionValue]
) -> pd.DataFrame:
    condition_df = pd.DataFrame()

    for condition_name, condition_value in spec_condition.items():
        condition_header = f"{DataframeHeaders.CONDITION} ({condition_name})"
        ranges = getattr(condition_value, "range", None) or []
        discrete = getattr(condition_value, "discrete", None) or None
        unit = getattr(condition_value, "unit", None)
        if len(ranges) > 0:
            condition_data = {
                (condition_header, DataframeHeaders.MIN): [range.min for range in ranges],
                (condition_header, DataframeHeaders.STEP): [range.step for range in ranges],
                (condition_header, DataframeHeaders.MAX): [range.max for range in ranges],
                (condition_header, DataframeHeaders.DISCRETE): [discrete for _ in ranges],
                (condition_header, DataframeHeaders.UNIT): [unit for _ in ranges],
            }
        else:
            condition_data = {
                (condition_header, DataframeHeaders.MIN): [None],
                (condition_header, DataframeHeaders.STEP): [None],
                (condition_header, DataframeHeaders.MAX): [None],
                (condition_header, DataframeHeaders.DISCRETE): [discrete],
                (condition_header, DataframeHeaders.UNIT): [unit],
            }
        spec_condition_df = pd.DataFrame(condition_data)
        condition_df = concatenate_dataframes([condition_df, spec_condition_df])

    return condition_df


# Builds the compliance dataframe of a spec from its measurement data.
def generate_spec_compliance_dataframe(
    spec: Specification,
    rows: int,
    measurement_data: List
) -> pd.DataFrame:
    min_compliance_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.MIN)
    max_compliance_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.MAX)
    mean_compliance_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.MEAN)
    median_compliance_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.MEDIAN)
    spec_health_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.HEALTH)
    spec_cpk_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.CPK)
    spec_ck_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.CP)
    spec_std_heading = (DataframeHeaders.COMPLIANCE, ComplianceParameters.STANDARD_DEVIATION)
    min_rows = max(1, rows)
    if len(measurement_data) > 0:
        measurement_df = pd.DataFrame({spec.spec_id: measurement_data})
        min_compliance = calculate_compliance(measurement_df, ComplianceParameters.MIN)
        max_compliance = calculate_compliance(measurement_df, ComplianceParameters.MAX)
        mean_compliance = calculate_compliance(measurement_df, ComplianceParameters.MEAN)
        median_compliance = calculate_compliance(measurement_df, ComplianceParameters.MEDIAN)
        spec_health = (
            get_spec_health(spec, str(min_compliance.iloc[0]), str(max_compliance.iloc[0]))
            if (str(min_compliance.iloc[0]) != SpecHealth.NAN and str(max_compliance.iloc[0])) != SpecHealth.NAN
            else str()
        )

        std = Decimal(measurement_df.astype('double').std(skipna=True).iloc[0])

        maxima = Decimal(max_compliance.iloc[0])
        minima = Decimal(min_compliance.iloc[0])
        mean = Decimal(mean_compliance.iloc[0])

        is_valid_max = bool(maxima)
        is_valid_min = bool(minima)
        cpk_high = (
            np.nan if not is_valid_max
            else __calculate_cpk_high(maxima, mean, std)
        )
        cpk_low = (
            np.nan if not is_valid_min 
            else __calculate_cpk_low(minima, mean, std)
        )
        cpk = min(cpk_high, cpk_low)
        
        cp = (
            np.nan if not (is_valid_max and is_valid_min)
            else __calculate_cp(maxima, minima, std)
        )

        compliance_data = {
            min_compliance_heading: [min_compliance.iloc[0]] * min_rows,
            max_compliance_heading: [max_compliance.iloc[0]] * min_rows,
            mean_compliance_heading: [mean_compliance.iloc[0]] * min_rows,
            median_compliance_heading: [median_compliance.iloc[0]] * min_rows,
            spec_health_heading: [spec_health] * min_rows,
            spec_std_heading: [std] * min_rows,
            spec_cpk_heading: [cpk] * min_rows,
            spec_ck_heading: [cp] * min_rows
        }
    else:
        empty_compliance_data = [None] * min_rows
        compliance_data = {
            min_compliance_heading: empty_compliance_data,
            max_compliance_heading: empty_compliance_data,
            mean_compliance_heading: empty_compliance_data,
            median_compliance_heading: empty_compliance_data,
            spec_health_heading: empty_compliance_data,
            spec_std_heading: empty_compliance_data,
            spec_cpk_heading: empty_compliance_data,
            spec_ck_heading: empty_compliance_data
        }
    compliance_df = pd.DataFrame(compliance_data)

    return compliance_df


# Builds the details and compliance dataframes of each spec, grouped by spec category.
def generate_spec_dataframe(specs: List[Specification], part_number: str) -> Dict:
    spec_data = {}
    spec_compliance_data = {}

    for spec in specs:
        category = spec.category if spec.category is not None else EMPTY_CATEGORY
        spec_condition = get_spec_condition(spec)
        spec_details_df = generate_spec_details_dataframe(spec)
        spec_conditions_df = generate_spec_conditions_dataframe(spec_condition)
        result_filter, steps_filter = get_steps_filters(spec, part_number)
        steps = query_steps(result_filter, steps_filter)
        measurement_data = fetch_measurement_data_for_spec(
            steps, steps_filter, spec_condition, spec.unit
        )
        spec_compliance_df = generate_spec_compliance_dataframe(
            spec, len(spec_conditions_df), measurement_data
        )
        spec_df = concatenate_dataframes([spec_details_df, spec_conditions_df])

        if category not in spec_data:
            spec_data[category] = spec_df
            spec_compliance_data[category] = spec_compliance_df
        else:
            spec_data[category] = concatenate_dataframes([spec_data[category], spec_df], False, 0)
            spec_compliance_data[category] = concatenate_dataframes(
                [spec_compliance_data[category], spec_compliance_df], False, 0
            )

    for category in spec_data:
        spec_data[category] = concatenate_dataframes(
            [spec_data[category], spec_compliance_data[category]]
        )

    return spec_data


### Update compliance properties on specs

Write the calculated metrics to each selected spec's custom properties. Updates are sent in batches, and failed updates are recorded with Scrapbook.


In [ ]:
# Splits a list into batches of the given size.
def __get_batches(list, batch_size):
    for i in range(0, len(list), batch_size):
        yield list[i:i+batch_size]


# Formats a compliance value to 4 decimal places.
def __format_compliance(value: Decimal) -> Decimal:
    return '{:.4f}'.format(value)


# Removes a compliance property from a spec if it exists.
def __delete_spec_custom_property(spec: Specification, property_to_be_deleted) -> None:
    if property_to_be_deleted in spec.properties:
        del spec.properties[property_to_be_deleted]


# Writes the compliance metrics to the custom properties of each spec and updates the specs.
def add_compliance_to_spec_custom_properties(specs: List[Specification], dataframe_dict: Dict) -> None:
    specs_to_be_updated = []
    for category, dataframe in dataframe_dict.items():
        specIds_df = dataframe[DataframeHeaders.SPEC_DETAILS][DataframeHeaders.SPEC_ID]
        for spec in specs:
            if spec.properties is None:
                spec.properties = {}
            spec_compliance_metrices_df = dataframe[specIds_df == spec.spec_id][DataframeHeaders.COMPLIANCE]
            
            if spec_compliance_metrices_df.empty or spec_compliance_metrices_df.iloc[0].empty or (not spec_compliance_metrices_df.iloc[0][ComplianceParameters.HEALTH]):
                __delete_spec_custom_property(spec, SpecComplianceProperties.MIN)
                __delete_spec_custom_property(spec, SpecComplianceProperties.MAX)
                __delete_spec_custom_property(spec, SpecComplianceProperties.MEAN)
                __delete_spec_custom_property(spec, SpecComplianceProperties.MEDIAN)
                __delete_spec_custom_property(spec, SpecComplianceProperties.HEALTH)
                __delete_spec_custom_property(spec, SpecComplianceProperties.STANDARD_DEVIATION)
                __delete_spec_custom_property(spec, SpecComplianceProperties.CPK)
                __delete_spec_custom_property(spec, SpecComplianceProperties.CP)

            else:
                spec.properties[SpecComplianceProperties.MIN] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.MIN]))
                spec.properties[SpecComplianceProperties.MAX] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.MAX]))
                spec.properties[SpecComplianceProperties.MEAN] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.MEAN]))
                spec.properties[SpecComplianceProperties.MEDIAN] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.MEDIAN]))
                spec.properties[SpecComplianceProperties.HEALTH] = str(spec_compliance_metrices_df.iloc[0][ComplianceParameters.HEALTH])
                
                spec.properties[SpecComplianceProperties.STANDARD_DEVIATION] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.STANDARD_DEVIATION]))
                spec.properties[SpecComplianceProperties.CPK] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.CPK]))
                spec.properties[SpecComplianceProperties.CP] = str(__format_compliance(spec_compliance_metrices_df.iloc[0][ComplianceParameters.CP]))
            
            specs_to_be_updated.append(spec)

    for specs_batch in __get_batches(specs_to_be_updated, MAX_SPECS_PER_UPDATE_REQUEST):
        update_specs_response = update_specs(specs_batch)
        if update_specs_response and update_specs_response.failed_specs:
            sb.glue(
                "Error updating spec custom properties with the compliance metrics",
                json.dumps(update_specs_response.model_dump(mode="json", by_alias=True)),
            )
            break


### Generate the Excel report

Create a worksheet for each spec category in the Excel report.

In [ ]:
# Writes the compliance dataframe of each category to its own Excel sheet.
def write_dataframe_to_excel(dataframe_dict: Dict, file_name: str) -> None:
    with pd.ExcelWriter(file_name, engine="openpyxl") as writer:
        for category, dataframe in dataframe_dict.items():
            dataframe.reset_index(drop=True, inplace=True)
            dataframe.to_excel(writer, sheet_name=f"Spec_{category}", index=True)


### Fetch specs, calculate compliance, and generate dataframes


In [ ]:
parametric_specs = query_parametric_specs(product_id)
spec_dataframe = generate_spec_dataframe(parametric_specs, part_number)

### Write compliance metrics to spec custom properties


In [ ]:
add_compliance_to_spec_custom_properties(parametric_specs, spec_dataframe)

### Write, upload, and link the Excel report


In [ ]:
pd.set_option("display.float_format", "{:.15f}".format)
write_dataframe_to_excel(spec_dataframe, EXCEL_FILE_NAME)
uploaded_file_id = upload_file(
    file_name=EXCEL_FILE_NAME,
    workspace_id=product_information.workspace,
)
product_update_response = add_file_id_to_product(
    product_information=product_information,
    file_id=uploaded_file_id,
)


### Store result information with Scrapbook

SystemLink uses Scrapbook to show the report link or update failures in the notebook execution details.


In [ ]:
if product_update_response.failed:
    sb.glue(
        "Error attaching the compliance report to the product",
        json.dumps(product_update_response.model_dump(mode="json", by_alias=True)),
    )
else:
    sb.glue(
        "Compliance Excel report generated and uploaded to the product",
        f'<a href="../../testinsights/products/product/{product_id}/files">Product Files tab</a>',
    )


### Limitations

- The notebook is designed to work only with the cases described in the rules above. Any other case may or may not work.
- If a spec has two or more conditions with the same name but different units (for example, `Vcc(V)` and `Vcc(I)`), the compliance might not be calculated properly. Only one of these conditions is shown in the compliance Excel report, and its data might not match the actual spec condition.

# Next Steps

1. Publish this notebook to SystemLink from the JupyterLab File Browser with the **Specification Analysis** interface.
1. Run it from the Specs grid in the product details page.
1. View the updated custom properties on the spec details page and the compliance report in the product's Files tab.
